# Chapitre 3 - Exercices 3.1 à 3.5

Ces exercices portent sur l'inférence dans le modèle linéaire (tests $Z$ et $F$, bandes de
confiance), le théorème de Gauss–Markov, le calcul des moindres carrés par Gram–Schmidt et
le rôle de l'ordonnée à l'origine dans ridge et le lasso.

Notations (cours 1) : $\mathbf{X}$ est la matrice $N\times(p+1)$ des entrées (première
colonne de uns), $\hat\beta = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$,
$\hat\sigma^2 = \mathrm{RSS}/(N-p-1)$, et $v_j$ le $j$-ième élément diagonal de
$(\mathbf{X}^T\mathbf{X})^{-1}$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import pandas as pd
from scipy import stats
from sklearn.linear_model import Lasso, Ridge
from esl import ch03_regression as cr

Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()

## Exercice 3.1 : Statistique $F$ et score $Z$

**Énoncé.** Montrer que la statistique $F$ (éq. (3.13)) associée au retrait d'**un seul**
coefficient du modèle est égale au carré du score $Z$ correspondant (éq. (3.12)).

**Rappels.** Pour un gros modèle à $p_1+1$ paramètres et un petit modèle emboîté à $p_0+1$
paramètres,

$$
F = \frac{(\mathrm{RSS}_0 - \mathrm{RSS}_1)/(p_1 - p_0)}{\mathrm{RSS}_1/(N - p_1 - 1)},
\qquad
z_j = \frac{\hat\beta_j}{\hat\sigma\sqrt{v_j}} .
$$

Ici $p_1 = p$, $p_0 = p - 1$, et le dénominateur de $F$ vaut $\hat\sigma^2$ (estimé dans le
gros modèle). Il suffit donc de prouver

$$
\mathrm{RSS}_0 - \mathrm{RSS}_1 = \frac{\hat\beta_j^2}{v_j}.
$$

**Étape 1 : la baisse de RSS due à $\mathbf{x}_j$.** Quitte à renuméroter, plaçons
$\mathbf{x}_j$ en **dernier**. Notons $\mathbf{X}_0$ la matrice sans $\mathbf{x}_j$ et
$\mathbf{z}_j = \mathbf{x}_j - \mathbf{H}_0\mathbf{x}_j$ le résidu de la régression de
$\mathbf{x}_j$ sur les autres colonnes ($\mathbf{H}_0$ = projecteur sur $\mathrm{col}(\mathbf{X}_0)$).
Alors $\mathrm{col}(\mathbf{X}) = \mathrm{col}(\mathbf{X}_0) \oplus \mathrm{vect}(\mathbf{z}_j)$,
somme **orthogonale** ($\mathbf{z}_j \perp \mathrm{col}(\mathbf{X}_0)$). Le projecteur sur une
somme orthogonale est la somme des projecteurs :

$$
\mathbf{H} = \mathbf{H}_0 + \frac{\mathbf{z}_j\mathbf{z}_j^T}{\lVert\mathbf{z}_j\rVert^2}.
$$

Comme $\mathrm{RSS} = \lVert(\mathbf{I} - \mathbf{H})\mathbf{y}\rVert^2 = \mathbf{y}^T(\mathbf{I} - \mathbf{H})\mathbf{y}$
(projecteur symétrique idempotent), on obtient

$$
\mathrm{RSS}_0 - \mathrm{RSS}_1 = \mathbf{y}^T(\mathbf{H} - \mathbf{H}_0)\mathbf{y}
  = \frac{\langle\mathbf{z}_j, \mathbf{y}\rangle^2}{\lVert\mathbf{z}_j\rVert^2}.
$$

**Étape 2 : lien avec $\hat\beta_j$ et $v_j$.** D'après l'éq. (3.28) (algorithme 3.1),
$\hat\beta_j = \langle\mathbf{z}_j, \mathbf{y}\rangle/\lVert\mathbf{z}_j\rVert^2$, et
d'après l'éq. (3.29), $\mathrm{Var}(\hat\beta_j) = \sigma^2/\lVert\mathbf{z}_j\rVert^2$. Or
l'éq. (3.8) donne aussi $\mathrm{Var}(\hat\beta_j) = \sigma^2 v_j$ ; donc
$v_j = 1/\lVert\mathbf{z}_j\rVert^2$. Ainsi

$$
\frac{\langle\mathbf{z}_j, \mathbf{y}\rangle^2}{\lVert\mathbf{z}_j\rVert^2}
  = \hat\beta_j^2\,\lVert\mathbf{z}_j\rVert^2 = \frac{\hat\beta_j^2}{v_j}.
$$

**Conclusion.**

$$
F = \frac{\hat\beta_j^2/v_j}{\hat\sigma^2} = \Bigl(\frac{\hat\beta_j}{\hat\sigma\sqrt{v_j}}\Bigr)^2 = z_j^2. \qquad\square
$$

C'est cohérent avec les lois : si $T \sim t_k$, alors $T^2 \sim F_{1,k}$.

**Vérification numérique** sur la prostate : pour chaque variable, on ajuste le modèle sans
elle et l'on compare $F$ à $z_j^2$.

In [2]:
full = cr.ols_fit(Xtr, ytr)
rows = []
for j in range(8):
    keep = [k for k in range(8) if k != j]
    F, pval = cr.f_test(cr.ols_fit(Xtr[:, keep], ytr)["rss"], full["rss"], 7, 8, len(ytr))
    rows.append((full["z"][j + 1] ** 2, F, pval, 2 * stats.t(full["df_resid"]).sf(abs(full["z"][j + 1]))))
pd.DataFrame(rows, index=names, columns=["z_j^2", "F", "p-valeur (F)", "p-valeur (t)"]).round(4)

,z_j^2,F,p-valeur (F),p-valeur (t)
lcavol,28.7971,28.7971,0.0000,0.0000
lweight,7.5668,7.5668,0.0079,0.0079
age,1.9486,1.9486,0.1681,0.1681
lbph,4.2265,4.2265,0.0443,0.0443
svi,6.0972,6.0972,0.0165,0.0165
lcp,3.4854,3.4854,0.0670,0.0670
gleason,0.0215,0.0215,0.8839,0.8839
pgg45,3.0201,3.0201,0.0875,0.0875
